# `02_Isolation_Forest.ipynb`

## 1. Intuition

Now we're moving from simple statistical rules to our first major anomaly-detection algorithm: **Isolation Forest**.

The most important question is:

> **Why should an anomaly be easier to isolate than a normal point?**

---

### Imagine points on a graph

Suppose most observations represent normal machine behavior:

```text id="m8k3p1"
Vibration
   ↑
   |
   |        ● ● ●
   |      ● ● ● ● ●
   |     ● ● ● ● ●
   |      ● ● ● ●
   |
   |                         ●
   |                         ↑
   |                      unusual
   +--------------------------------→ Temperature
```

Most points form a large group.

One point is far away.

Now imagine that instead of calculating distances, we play a game:

> **Randomly cut the space until a particular point is separated from all the other points.**

For the isolated point, we might need only a few cuts:

```text id="f7j2qk"
All points
   ↓
Random cut
   ↓
Most points | isolated point
                  ↓
                DONE
```

But for a point sitting inside the dense group:

```text id="q9m4xz"
Dense group
   ↓
Random cut
   ↓
Still many points together
   ↓
Another random cut
   ↓
Still many points together
   ↓
Another random cut
   ↓
...
   ↓
Eventually isolated
```

So:

$$
\boxed{\text{Anomalies tend to be isolated with fewer random splits}}
$$

while:

$$
\boxed{\text{Normal points tend to require more splits}}
$$

This is the **central intuition behind Isolation Forest**.

---

## Why is this useful?

Notice that we aren't asking:

> "How far is this point from the mean?"

We aren't asking:

> "Is its density low?"

Instead, we're asking:

> **"How quickly can random partitions separate this point from the rest of the data?"**

That gives us a completely different definition of unusualness.

---

## Why does random splitting work?

Consider two situations.

### Normal point

A normal point sits inside a dense region:

```text id="3n6v8c"
● ● ● ● ●
● ● X ● ●
● ● ● ● ●
```

The point \(X\) has many neighboring points around it.

A random split is likely to leave \(X\) together with many other observations.

So we need several splits before \(X\) becomes isolated.

---

### Anomalous point

Now consider:

```text id="9q2m5x"
● ● ● ● ●

                    X
```

There are very few points around \(X\).

A random split has a much higher chance of separating \(X\) from the main group quickly.

Therefore:

$$
\boxed{
\text{Short isolation path}
\rightarrow
\text{Likely anomaly}
}
$$

$$
\boxed{
\text{Long isolation path}
\rightarrow
\text{Likely normal}
}
$$

---

## Why is it called a "Forest"?

Because we don't create just **one random tree**.

We create **many isolation trees**.

```text id="m2c8vz"
                    Data
                      ↓
        ┌─────────────┼─────────────┐
        ↓             ↓             ↓
     Tree 1        Tree 2        Tree 3
        ↓             ↓             ↓
     Path           Path           Path
        ↓             ↓             ↓
        └─────────────┼─────────────┘
                      ↓
              Combine results
                      ↓
                Anomaly Score
```

This collection of trees is the **Isolation Forest**.

Why use many trees?

Because one random tree can give an unreliable result.

For example, an anomaly might happen to require several unlucky splits in one tree.

By generating many different random trees, we get a more stable estimate of how easily the point can be isolated.

---

## The complete intuition

The algorithm can therefore be understood as:

$$
\boxed{
\text{Randomly partition data}
\rightarrow
\text{Measure how quickly each point becomes isolated}
\rightarrow
\text{Shorter path}
\rightarrow
\text{More anomalous}
}
$$

This is the key idea you should remember before we get into the actual tree construction.

### One important distinction

Isolation Forest does **not** literally search for "the point farthest away."

A point can be anomalous because it is:

* far away from the main group,
* in a sparse region,
* or otherwise easy to separate through random partitions.

So its strength is that it detects anomalies through **isolation**, rather than explicitly calculating distance or density.


## 2. Why Anomalies Are Easier to Isolate

Now let's understand the **reason** behind the Isolation Forest idea.

The whole algorithm depends on one observation:

> **Anomalies are usually few and different from the majority of observations.**

Because of that, random splits tend to separate them quickly.

---

### Imagine a 1D dataset

Suppose our points are:

$$
2,\ 3,\ 4,\ 5,\ 6,\ 7,\ 8,\ 20
$$

Most points are between:

$$
2 \rightarrow 8
$$

while:

$$
20
$$

is isolated from the main group.

Imagine choosing a random split somewhere between 2 and 20.

Suppose we randomly choose:

$$
10
$$

Then:

```text id="q4r8z2"
2  3  4  5  6  7  8 | 20
                        ↑
                    isolated
```

One split was enough to separate 20 from every other observation.

So its **path length = 1** in this particular tree.

---

### Now consider a normal point

Take:

$$
5
$$

A random split at 10 gives:

```text id="w3k7m1"
2  3  4  5  6  7  8 | 20
      ↑
    still with many points
```

We haven't isolated 5.

We need another split.

Suppose we split at 4:

```text id="a8p2x6"
2  3 | 4  5  6  7  8 | 20
        ↑
      still with others
```

Another split might be needed:

```text id="r5m9c3"
4 | 5  6  7  8
    ↑
```

And eventually:

```text id="z7k1q4"
4 | 5 | 6  7  8
    ↑
   isolated
```

So the normal point takes **more splits** to isolate.

---

## The key reason

Look at the two situations.

### Anomaly

```text id="h2v6p9"
Main group                    Anomaly

● ● ● ● ● ● ● ●                  ●
                                  ↑
                              very few
                              nearby points
```

There are very few observations around the anomaly.

Therefore, many possible random cuts can separate it quickly.

---

### Normal point

```text id="n4c8y2"
        Dense region

     ● ● ● ● ●
    ● ● X ● ●
     ● ● ● ● ●
```

There are many observations around \(X\).

A random split is likely to keep \(X\) together with other points.

Therefore, more splits are generally required.

---

# The important probability intuition

Suppose a point has many neighboring observations.

A random split has to be **quite specific** to isolate that point.

But if a point is already sitting alone in a sparse region, many different split locations can isolate it.

Conceptually:

$$
\boxed{
\text{Few nearby points}
\rightarrow
\text{Many possible isolating splits}
\rightarrow
\text{Short path}
}
$$

while:

$$
\boxed{
\text{Many nearby points}
\rightarrow
\text{Fewer useful isolating splits}
\rightarrow
\text{Long path}
}
$$

That's why the method doesn't need to explicitly calculate distance.

---

## What if the anomaly isn't far away?

This is an important case.

Consider:

```text id="c5x8n1"
● ● ● ● ● ● ●
● ● ● ● ●
● ● ● ● ●
        ●
```

The unusual point isn't necessarily extremely far away.

But if it lies in a relatively sparse region, random partitions can still isolate it faster than points deep inside the dense cluster.

So Isolation Forest is not simply:

$$
\text{Far away} \rightarrow \text{Anomaly}
$$

Instead:

$$
\boxed{
\text{Easy to isolate}
\rightarrow
\text{Potential anomaly}
}
$$

---

# Why one tree isn't enough

Here's another important point.

Suppose we build one random tree.

It might happen that the random splits are unlucky:

```text id="g1p6w4"
Anomaly
   ↓
unlucky random splits
   ↓
takes longer to isolate
```

That single tree might therefore give us a misleading result.

So we build **many trees**:

```text id="k3z9m2"
                 Data
                   ↓
       ┌───────────┼───────────┐
       ↓           ↓           ↓
    Tree 1      Tree 2      Tree 3
       ↓           ↓           ↓
    Path=2       Path=3       Path=2
       │           │           │
       └───────────┼───────────┘
                   ↓
             Average behavior
                   ↓
             Anomaly score
```

If a point consistently gets **short paths across many random trees**, we have stronger evidence that it is anomalous.

---

## Core idea

Everything in Isolation Forest follows from this:

$$
\boxed{
\text{Anomalies are rare and different}
\rightarrow
\text{They are easier to isolate}
\rightarrow
\text{They have shorter paths in isolation trees}
}
$$

The next important question is:

> **How exactly do we construct one of these isolation trees?**


## 3. Isolation Trees

Now we'll see **how the tree is actually constructed**.

You can think of an Isolation Tree as a decision tree whose only purpose is:

> **Keep randomly splitting the data until individual observations become isolated.**

It is different from a normal supervised decision tree because we are **not predicting a target/class**.

---

### 3.1 Start with all the data

Consider this simple 1D dataset:

$$
2,\ 3,\ 4,\ 5,\ 6,\ 7,\ 20
$$

Initially, all observations are together:

```text id="q3v8m1"
[2, 3, 4, 5, 6, 7, 20]
```

This is the **root node** of the tree.

---

### 3.2 Make a random split

The algorithm chooses a split value randomly between the minimum and maximum values.

Suppose it randomly chooses:

$$
s=10
$$

The rule becomes:

$$
x<10
$$

go to the left branch, while:

$$
x\geq10
$$

go to the right branch.

So:

```text id="m8k2p4"
                    [2,3,4,5,6,7,20]
                           │
                    split at 10
                      /         \
                     /           \
                    ↓             ↓
          [2,3,4,5,6,7]          [20]
                                  ↑
                              isolated!
```

The point 20 has become isolated after just one split.

---

## 3.3 What happens to the larger group?

The left node still contains:

$$
[2,3,4,5,6,7]
$$

So we continue splitting **only this node**.

Suppose the next random split is:

$$
s=4.5
$$

Then:

```text id="z1c7n5"
              [2,3,4,5,6,7]
                     │
                  4.5
                 /   \
                /     \
               ↓       ↓
          [2,3,4]    [5,6,7]
```

Neither group contains just one observation yet.

So we continue.

---

### 3.4 Continue recursively

Suppose the left node:

$$
[2,3,4]
$$

gets split at:

$$
s=2.5
$$

Then:

```text id="w6r2k8"
             [2,3,4]
                │
              2.5
             /   \
            /     \
           ↓       ↓
         [2]      [3,4]
          ↑
       isolated
```

Now 2 is isolated.

The process continues on \([3,4]\).

Eventually we might obtain a tree like:

```text id="t9m4q2"
                       [2,3,4,5,6,7,20]
                              │
                           split=10
                         /          \
                        /            \
               [2,3,4,5,6,7]       [20]
                    │                 ↑
                 split=4.5        isolated
                  /     \
                 /       \
            [2,3,4]     [5,6,7]
               │            │
            split=2.5    ...
             /    \
           [2]   [3,4]
            ↑
         isolated
```

This is an **Isolation Tree**.

---

# 3.5 What makes it different from a normal Decision Tree?

A normal supervised decision tree might look like:

```text
Features
   ↓
Choose split that improves prediction
   ↓
Predict class/value
```

For example:

$$
\text{Income}<50,000
\rightarrow
\text{Low Risk}
$$

The split is chosen based on some objective such as impurity reduction.

Isolation Trees work differently:

```text
Data
 ↓
Random feature
 ↓
Random split
 ↓
Partition
 ↓
Repeat
 ↓
Isolate observations
```

There is **no target variable \(y\)** being predicted.

The tree exists purely to **separate observations**.

---

# 3.6 When does the tree stop?

The recursive splitting doesn't continue forever.

A node can stop when, for example:

### Case 1 — Only one observation remains

```text id="r4v8m2"
[20]
 ↓
STOP
```

It has been isolated.

---

### Case 2 — All observations have identical values

Suppose:

$$
[5,5,5,5]
$$

There is no useful random split between different values.

So the node cannot be meaningfully divided further.

---

### Case 3 — Maximum tree depth is reached

The algorithm can impose a maximum depth so that trees don't grow unnecessarily large.

We'll discuss this parameter later under **important hyperparameters**.

---

# 3.7 Where does the anomaly idea enter?

Look at our example again:

```text id="f1m7q3"
20
 ↓
isolated after 1 split
```

while:

```text id="k8p2v6"
5
 ↓
several splits
 ↓
eventually isolated
```

So each observation ends up with a **path from the root of the tree to the point where it becomes isolated**.

For example:

$$
20 \rightarrow \text{short path}
$$

$$
5 \rightarrow \text{longer path}
$$

This leads directly to the next important concept:

$$
\boxed{\text{Path Length}}
$$

The path length will allow us to turn the tree's behavior into a numerical measure of how unusual each observation is.


## 4. Random Feature Selection

So far, we used a **1-dimensional example**, so there was only one feature to choose from.

Real datasets usually have multiple features.

For example, a machine could have:

| Point | Temperature | Vibration | Pressure |
| ----- | ----------: | --------: | -------: |
| A     |          50 |        20 |      100 |
| B     |          51 |        21 |      102 |
| C     |          49 |        19 |       99 |
| D     |          90 |        80 |      150 |

Now the Isolation Forest has to decide:

> **Which feature should I use to make the next split?**

### Intuition

Isolation Forest does **not** search for the "best" feature like a normal supervised decision tree.

Instead, it chooses a feature **randomly**.

For example:

```text id="v8q3m1"
Features
   │
   ├── Temperature
   ├── Vibration
   └── Pressure
          ↓
     Randomly choose
          ↓
       Vibration
          ↓
    Make random split
```

Why random?

Because the goal isn't to make the best prediction.

The goal is to create **random partitions** and see how quickly observations can be isolated.

---

## Why can random feature selection isolate anomalies?

Look at our example:

```text id="n4k7p2"
Normal points

Temperature   Vibration
    50           20
    51           21
    49           19
```

and:

```text id="c9m2x5"
Anomaly

Temperature = 90
Vibration   = 80
```

The anomaly is unusual in **both features**.

So if the algorithm randomly chooses:

$$
\text{Temperature}
$$

it has a good opportunity to isolate the point.

If it randomly chooses:

$$
\text{Vibration}
$$

it also has a good opportunity.

This is useful because we don't need to manually tell the algorithm:

> "Temperature is the important feature."

The random process discovers isolation opportunities naturally.

---

## Tiny Example

Suppose we have:

$$
\begin{bmatrix}
10 & 20 \\
11 & 21 \\
12 & 19 \\
50 & 80
\end{bmatrix}
$$

The columns are:

* Feature 1 = Temperature
* Feature 2 = Vibration

Suppose the algorithm randomly selects **Feature 2**.

The values are:

$$
20,\ 21,\ 19,\ 80
$$

It might randomly choose a split:

$$
s=50
$$

Then:

$$
x_2<50
$$

gives:

$$
20,\ 21,\ 19
$$

and:

$$
x_2\geq50
$$

gives:

$$
80
$$

So the anomalous observation is isolated immediately.

---

## What if the randomly selected feature doesn't help?

Suppose instead the algorithm chooses a feature where the anomaly isn't particularly different.

For example:

```text id="a6p3k9"
Feature 3

Normal:   100, 101, 99
Anomaly:  102
```

A random split might not isolate the anomaly quickly.

That's okay.

Remember:

$$
\boxed{\text{One tree is random}}
$$

We don't rely on one tree.

Isolation Forest creates **many trees with different random feature selections and random splits**.

Some trees will isolate the observation quickly.

Others may take longer.

We combine their results.

```text id="j7q4m8"
                    Data
                      ↓
          ┌───────────┼───────────┐
          ↓           ↓           ↓
       Tree 1      Tree 2      Tree 3
       Feature 1   Feature 2   Feature 3
          ↓           ↓           ↓
       Path = 2    Path = 4    Path = 2
          └───────────┼───────────┘
                      ↓
               Combine paths
                      ↓
                Final score
```

This randomness is actually a **strength**, not a weakness.

---

## Important distinction

A normal decision tree asks:

> **"Which feature and split give me the best prediction?"**

Isolation Forest asks:

> **"Let's randomly choose a feature and split, then see how quickly observations become isolated."**

So:

$$
\boxed{
\text{Decision Tree}
\rightarrow
\text{Best feature/split}
}
$$

while:

$$
\boxed{
\text{Isolation Tree}
\rightarrow
\text{Random feature/split}
}
$$

This randomization is fundamental to why the method is called an **Isolation Forest**.


Sure. Let's take a **very small dataset with 3 features** and build one Isolation Tree manually.

### Dataset

Suppose we have 6 points:

| Point | Feature 1 | Feature 2 | Feature 3 |
| ----- | --------: | --------: | --------: |
| A     |         2 |        10 |       100 |
| B     |         3 |        11 |       101 |
| C     |         4 |        10 |        99 |
| D     |         5 |        12 |       102 |
| E     |         6 |        11 |       100 |
| F     |        20 |        30 |       150 |

Here **F looks unusual** compared with A–E.

---

## Step 1: Root node

We start with all 6 points:

```text
A B C D E F
```

Randomly select a feature.

Suppose:

```text
Random Feature = Feature 1
```

Feature 1 values:

```text
A  2
B  3
C  4
D  5
E  6
F 20
```

Suppose the random split value is:

```text
Feature 1 < 13
```

So:

```text
                 Root
            Feature 1 < 13?
              /          \
            YES            NO
         A B C D E          F
```

F is **already isolated**.

So F has a very short path.

---

But let's make the example more interesting and follow the **same tree** where features can change and even repeat.

Suppose instead the first random split was:

```text
Feature 1 < 4.5
```

Then:

```text
                    Root
               Feature 1 < 4.5?
                  /          \
                YES            NO
              A B C          D E F
```

Now we go into the right child:

```text
D E F
```

### Step 2: New node

We **randomly select a feature again**.

Suppose:

```text
Random Feature = Feature 3
```

Feature 3 values for D, E, F:

```text
D → 102
E → 100
F → 150
```

Suppose random split:

```text
Feature 3 < 125
```

Now:

```text
                    Root
               Feature 1 < 4.5?
                  /          \
                YES            NO
              A B C          D E F
                              |
                         Feature 3 < 125?
                           /          \
                         D E           F
```

F is isolated.

---

### Now notice what happened

Our tree used:

```text
Root       → Feature 1
Next node  → Feature 3
```

It **could** have selected Feature 1 again at the second node.

For example:

```text
                    Root
               Feature 1 < 4.5?
                  /          \
                YES            NO
              A B C          D E F
                              |
                         Feature 1 < 10?
                           /          \
                         D E           F
```

Here **Feature 1 was selected twice in the same tree**.

And it is also possible to have:

```text
Feature 1
   ↓
Feature 3
   ↓
Feature 1
   ↓
Feature 2
   ↓
Feature 1
```

There is **no restriction on reusing a feature**.

The important rule is simply:

```text
At each node
      ↓
Randomly choose a feature
      ↓
Randomly choose a split value
      ↓
Split the data
      ↓
Repeat for the child nodes
```

That is the key mechanism of the Isolation Tree.


Yes — **with one important correction**.

### 1. If one feature value is unusual

An Isolation Forest works on the **entire row/observation**, not on an individual feature independently.

For example:

| Person |    Age | Income | Spending |
| ------ | -----: | -----: | -------: |
| A      |     21 |    50k |      20k |
| B      |     22 |    52k |      21k |
| C      |     23 |    51k |      19k |
| D      | **80** |    50k |      20k |

Here, `Age = 80` is unusual.

So **Person D (the entire row) can be identified as an anomaly** because that unusual feature makes the row easy to isolate.

```text
Feature 1 (Age)
      ↓
80 is far from other values
      ↓
Random split may isolate D quickly
      ↓
Short path length
      ↓
D gets high anomaly score
      ↓
Entire observation D = anomaly
```

But an important point:

> **One unusual feature does not automatically mean the row is an anomaly.**

The forest looks at how easily the **whole observation** gets isolated across many random trees.

---

### 2. And yes, we build multiple trees 🌳🌳🌳

Exactly.

For example:

```text
                 DATA
                   │
       ┌───────────┼───────────┐
       ↓           ↓           ↓
    Tree 1      Tree 2      Tree 3
       ↓           ↓           ↓
   isolate      isolate      isolate
    points       points       points
       │           │           │
       └───────────┼───────────┘
                   ↓
          Combine path lengths
                   ↓
          Anomaly Score
                   ↓
          Normal / Anomaly
```

Each tree is **randomly different**:

```text
Tree 1:
Feature 1 → Feature 3 → Feature 1 → ...

Tree 2:
Feature 2 → Feature 1 → Feature 2 → ...

Tree 3:
Feature 3 → Feature 3 → Feature 1 → ...
```

The same point might be isolated:

```text
Tree 1 → path length 3
Tree 2 → path length 4
Tree 3 → path length 3
Tree 4 → path length 5
...
```

If it **consistently gets short paths across many trees**, the forest says:

> "This point is easy to isolate → likely anomaly."

If it generally needs **long paths**, it is considered more normal.

So the overall idea is:

**Random feature → random split → isolation tree → many trees → average isolation behavior → anomaly score → decision.**


## 6. Path Length

Now we have the tree and understand how the splitting happens.

The next question is:

> **How does Isolation Forest decide whether a point was isolated quickly or slowly?**

It uses **path length**.

### What is path length?

Path length = **number of splits a point goes through from the root until it becomes isolated.**

For example:

```text
                 Root
                   |
             F1 < 20?
              /     \
             /       \
          A-I         J
                     ↑
                 isolated
```

For `J`:

```text
Root
 ↓
F1 < 20?
 ↓
J
```

J was isolated after **1 split**.

So:

$$
h(J)=1
$$

where:

* \(h(J)\) = path length of point \(J\)
* \(1\) = number of splits needed to isolate it

---

### Compare with a normal point

Suppose another point needs several splits:

```text
Root
 ↓
F1 < 20?
 ↓
A B C D E F
 ↓
F2 < 21?
 ↓
B C D
 ↓
F3 < 100?
 ↓
C
```

Point C needed **3 splits**.

Therefore:

$$
h(C)=3
$$

So we have:

| Point | Path length |
| ----- | ----------: |
| J     |           1 |
| C     |           3 |

And the fundamental Isolation Forest idea is:

$$
\boxed{\text{Short path} \rightarrow \text{more likely anomaly}}
$$

$$
\boxed{\text{Long path} \rightarrow \text{more likely normal}}
$$

### Why?

Because an unusual point is usually **far from the dense group**.

Therefore, random splits can separate it quickly.

A normal point is surrounded by other similar points, so random splitting has to keep dividing the group before that point becomes alone.

```text
ANOMALY                         NORMAL

   J                          A B C D E
   |                           |
split 1                     split 1
   |                           |
   J                        A B C
                               |
                            split 2
                               |
                              A B
                               |
                            split 3
                               |
                               A
```

So:

```text
Anomaly → short path
Normal  → long path
```

But remember: **one tree is random**. A point might get a short path by chance.

That's exactly why we build a **forest of many trees** and later combine the path lengths to calculate the **anomaly score**.


## 7. Anomaly Score

We now know that **shorter path length means more anomalous**.

But one tree is random. So we need to combine the path lengths from **many trees** into one number.

That number is the **anomaly score**.

### Step 1: Take one point

Suppose we have point `J`.

We build 5 isolation trees:

| Tree   | Path length of J |
| ------ | ---------------: |
| Tree 1 |                2 |
| Tree 2 |                3 |
| Tree 3 |                2 |
| Tree 4 |                4 |
| Tree 5 |                3 |

Average path length:

$$
\frac{2+3+2+4+3}{5}=2.8
$$

So, on average, J gets isolated after **2.8 splits**.

---

### Step 2: Compare with a normal point

Suppose point `C` has:

| Tree   | Path length |
| ------ | ----------: |
| Tree 1 |           6 |
| Tree 2 |           7 |
| Tree 3 |           6 |
| Tree 4 |           8 |
| Tree 5 |           7 |

Average:

$$
\frac{6+7+6+8+7}{5}=6.8
$$

So:

```text
J → average path = 2.8 → isolated quickly
C → average path = 6.8 → needs many splits
```

Therefore:

$$
\boxed{J\text{ is more likely to be anomalous}}
$$

---

### But how does this become an actual score?

Isolation Forest uses a **normalized anomaly score**, rather than simply saying:

> average path = 2.8 → anomaly

Conceptually:

$$
\boxed{\text{Shorter average path} \rightarrow \text{higher anomaly score}}
$$

$$
\boxed{\text{Longer average path} \rightarrow \text{lower anomaly score}}
$$

The path length is normalized using the expected path length of a randomly constructed isolation tree.

The important idea for now is:

```text
              Many Trees
                  ↓
       Path length for each point
                  ↓
        Average path length
                  ↓
        Normalize the result
                  ↓
          Anomaly Score
                  ↓
       ┌──────────┴──────────┐
       ↓                     ↓
  More anomalous          More normal
```

### One important distinction

Don't confuse:

**Path length** with **anomaly score**.

They move in opposite directions:

| Average path | Interpretation   | Anomaly score |
| -----------: | ---------------- | ------------- |
|        Short | Easy to isolate  | High          |
|       Medium | Somewhat unusual | Medium        |
|         Long | Hard to isolate  | Low           |

So the complete logic is:

$$
\boxed{
\text{Random splits}
\rightarrow
\text{Path lengths}
\rightarrow
\text{Average path length}
\rightarrow
\text{Anomaly score}
\rightarrow
\text{Anomaly decision}
}
$$

The next important part is to see **exactly how the mathematical anomaly-score formula is constructed from the path length**.


## 8. Anomaly Score — Mathematical Formula

Now let's go one level deeper and see **how the path length is converted into an anomaly score**.

### 1. First, the average path length

Suppose we build \(T\) isolation trees.

For a point \(x\), let:

$$
h_t(x)
$$

be the path length of \(x\) in tree \(t\).

Then the average path length is:

$$
E[h(x)] =
\frac{1}{T}
\sum_{t=1}^{T} h_t(x)
$$

Where:

* \(T\) = number of trees
* \(h_t(x)\) = path length of point \(x\) in tree \(t\)
* \(E[h(x)]\) = average path length across the forest

For example:

$$
h_1(J)=2,\quad
h_2(J)=3,\quad
h_3(J)=2,\quad
h_4(J)=4,\quad
h_5(J)=3
$$

Therefore:

$$
E[h(J)]
=
\frac{2+3+2+4+3}{5}
=
2.8
$$

---

## 2. We need to normalize this

Here's the problem.

A path length of `3` doesn't mean the same thing for every dataset.

Imagine:

```text
Dataset A → 10 points
Dataset B → 10,000 points
```

A point isolated in 3 splits is very different depending on the dataset size.

So Isolation Forest compares the observed path length against the **expected path length of an unsuccessful search in a random binary search tree**.

This normalization factor is called:

$$
c(n)
$$

where \(n\) is the number of points in the node/sample.

For \(n>2\):

$$
c(n)
=
2H(n-1)
-
\frac{2(n-1)}{n}
$$

where \(H(n-1)\) is the harmonic number:

$$
H(n-1)
=
\sum_{i=1}^{n-1}\frac{1}{i}
$$

You don't need to memorize this formula right now. Its purpose is simply to give us a **reference path length**.

---

## 3. The anomaly score

The original Isolation Forest formulation uses:

$$
s(x,n)
=
2^{-\frac{E[h(x)]}{c(n)}}
$$

Now notice something important.

The average path length is in the **negative exponent**:

$$
-\frac{E[h(x)]}{c(n)}
$$

Therefore:

### Short path

Suppose:

$$
E[h(x)] \text{ is small}
$$

Then the exponent is closer to \(0\), so:

$$
s(x,n) \rightarrow 1
$$

Meaning:

$$
\boxed{\text{High score} \rightarrow \text{more anomalous}}
$$

### Long path

If:

$$
E[h(x)] \text{ is large}
$$

then the exponent becomes more negative:

$$
s(x,n) \rightarrow 0
$$

Meaning:

$$
\boxed{\text{Low score} \rightarrow \text{more normal}}
$$

---

### The whole idea

```text
        Point X
           ↓
    Build many trees
           ↓
   Find path length
   in every tree
           ↓
 Average path length
           ↓
      Normalize
           ↓
    Anomaly Score
           ↓
   ┌───────┴────────┐
   ↓                ↓
 score closer     score closer
 to 1             to 0
   ↓                ↓
anomalous          normal
```

So mathematically:

$$
\boxed{
\text{Shorter path}
\rightarrow
\text{smaller }E[h(x)]
\rightarrow
\text{higher }s(x,n)
\rightarrow
\text{more anomalous}
}
$$

One caveat: **scikit-learn's `score_samples()` uses a sign convention where more negative values indicate more abnormal observations**, so its returned score is not the same number/sign convention as the original \(s(x,n)\) formula above. This distinction becomes important when we implement it in Python.


Yes. Let's do the **entire Isolation Forest numerical example from beginning to final anomaly decision**, without skipping the calculation.

We will use **10 rows and 3 features**, build several trees, calculate path lengths, calculate \(c(n)\), calculate anomaly scores, and finally compare the points.

---

# Complete Numerical Example

## 1. Dataset

Consider this dataset:

| Point |     F1 |     F2 |      F3 |
| ----- | -----: | -----: | ------: |
| A     |     10 |     20 |     100 |
| B     |     11 |     21 |     101 |
| C     |     12 |     19 |      99 |
| D     |     13 |     22 |     102 |
| E     |     14 |     20 |     100 |
| F     |     15 |     23 |     103 |
| G     |     11 |     20 |      98 |
| H     |     13 |     21 |     101 |
| I     |     14 |     22 |     102 |
| **J** | **30** | **50** | **150** |

We intentionally made J unusual.

Notice that J is unusual in **all three features**:

$$
F1=30,\qquad F2=50,\qquad F3=150
$$

while most other points are clustered around:

$$
F1\approx10-15
$$

$$
F2\approx19-23
$$

$$
F3\approx98-103
$$

---

# 2. Build Isolation Trees

Suppose our forest contains **5 trees**.

Each tree randomly chooses:

1. a feature
2. a split value

and recursively splits the resulting groups.

---

## Tree 1

Randomly choose:

$$
\boxed{F1}
$$

Random threshold:

$$
\boxed{20}
$$

Therefore:

$$
F1<20
$$

gives:

```text
                F1 < 20?
                /       \
               /         \
          A B C D E F G H I    J
                                ↑
                             isolated
```

J has:

$$
F1=30
$$

Therefore J goes into a node containing only itself.

So:

$$
\boxed{h_1(J)=1}
$$

---

# 3. Tree 2

This time the random feature is different.

Choose:

$$
\boxed{F2}
$$

Random threshold:

$$
\boxed{35}
$$

So:

$$
F2<35
$$

gives:

```text
                F2 < 35?
                /       \
               /         \
          A B C D E F G H I    J
                                ↑
                             isolated
```

J has:

$$
F2=50
$$

So again:

$$
\boxed{h_2(J)=1}
$$

---

# 4. Tree 3

Randomly choose:

$$
\boxed{F3}
$$

Random threshold:

$$
\boxed{120}
$$

Then:

```text
                F3 < 120?
                /        \
               /          \
          A B C D E F G H I    J
                                ↑
                             isolated
```

J has:

$$
F3=150
$$

Therefore:

$$
\boxed{h_3(J)=1}
$$

---

# 5. Tree 4

Now let's make the tree more interesting.

First random choice:

$$
\boxed{F1<12}
$$

So:

```text
                    F1 < 12?
                   /         \
                A B G       C D E F H I J
```

J is **not isolated** yet.

So we continue splitting the right branch.

Suppose the next random choice is:

$$
\boxed{F2<30}
$$

Then:

```text
                    F1 < 12?
                   /         \
                A B G       C D E F H I J
                              |
                           F2 < 30?
                           /       \
                    C D E F H I     J
                                    ↑
                                 isolated
```

J is now alone.

It required **2 splits**:

$$
\boxed{h_4(J)=2}
$$

---

# 6. Tree 5

Suppose the tree starts with:

$$
\boxed{F3<110}
$$

Then:

```text
                    F3 < 110?
                   /          \
                  /            \
          A B C D E F G H I      J
                                ↑
                             isolated
```

Therefore:

$$
\boxed{h_5(J)=1}
$$

---

# 7. Path lengths of J

Now we have:

| Tree   | Path length |
| ------ | ----------: |
| Tree 1 |           1 |
| Tree 2 |           1 |
| Tree 3 |           1 |
| Tree 4 |           2 |
| Tree 5 |           1 |

Average path length:

$$
E[h(J)]
=
\frac{1+1+1+2+1}{5}
$$

$$
\boxed{E[h(J)]=1.2}
$$

So J is isolated, on average, after only **1.2 splits**.

---

# 8. Now calculate the reference path length \(c(n)\)

This is the part you specifically asked about earlier.

We have:

$$
n=10
$$

The normalization factor is:

$$
c(n)
=
2H(n-1)-\frac{2(n-1)}{n}
$$

For \(n=10\):

$$
c(10)
=
2H(9)-\frac{18}{10}
$$

Now calculate \(H(9)\):

$$
H(9)
=
1+\frac12+\frac13+\frac14+\frac15+\frac16+\frac17+\frac18+\frac19
$$

$$
H(9)\approx2.829
$$

Therefore:

$$
c(10)
=
2(2.829)-1.8
$$

$$
c(10)
=
5.658-1.8
$$

$$
\boxed{c(10)\approx3.858}
$$

### What does \(3.858\) mean?

It is our **reference expected path length** for a random unsuccessful search with 10 points.

So we now compare:

$$
\boxed{J\text{'s observed average path}=1.2}
$$

against:

$$
\boxed{\text{reference path}=3.858}
$$

And clearly:

$$
1.2<3.858
$$

J is being isolated **much faster than the reference**.

---

# 9. Calculate J's anomaly score

The original Isolation Forest score is:

$$
s(x,n)
=
2^{-\frac{E[h(x)]}{c(n)}}
$$

For J:

$$
s(J,10)
=
2^{-\frac{1.2}{3.858}}
$$

First:

$$
\frac{1.2}{3.858}\approx0.311
$$

Therefore:

$$
s(J,10)
=
2^{-0.311}
$$

$$
\boxed{s(J,10)\approx0.806}
$$

So J has an anomaly score of approximately:

$$
\boxed{0.806}
$$

---

# 10. Now let's calculate for a normal point C

We calculate C **only because we want to compare a normal-looking point with J**.

Suppose across our 5 trees, C gets:

| Tree   | Path length |
| ------ | ----------: |
| Tree 1 |           6 |
| Tree 2 |           5 |
| Tree 3 |           7 |
| Tree 4 |           6 |
| Tree 5 |           7 |

Average:

$$
E[h(C)]
=
\frac{6+5+7+6+7}{5}
$$

$$
\boxed{E[h(C)]=6.2}
$$

Now compare with the same reference:

$$
c(10)=3.858
$$

C's path is:

$$
6.2>3.858
$$

So C requires considerably more splitting than the reference.

---

# 11. C's anomaly score

$$
s(C,10)
=
2^{-\frac{6.2}{3.858}}
$$

Calculate:

$$
\frac{6.2}{3.858}\approx1.607
$$

Therefore:

$$
s(C,10)
=
2^{-1.607}
$$

$$
\boxed{s(C,10)\approx0.328}
$$

---

# 12. Final comparison

| Point | Average path \(E[h(x)]\) | Reference \(c(10)\) | Anomaly score |
| ----- | -----------------------: | ------------------: | ------------: |
| **J** |                  **1.2** |               3.858 |     **0.806** |
| **C** |                  **6.2** |               3.858 |     **0.328** |

Now everything becomes clear:

```text
                 Reference
                 c(10)=3.858
                     │
          ┌──────────┴──────────┐
          ↓                     ↓
       J = 1.2              C = 6.2
          ↓                     ↓
   much shorter             much longer
          ↓                     ↓
   score = 0.806           score = 0.328
          ↓                     ↓
     anomalous                normal
```

### The complete chain

$$
\boxed{
\text{Dataset}
\rightarrow
\text{Random feature}
\rightarrow
\text{Random threshold}
\rightarrow
\text{Recursive splitting}
\rightarrow
\text{Path length}
\rightarrow
\text{Average path length}
\rightarrow
c(n)
\rightarrow
\text{Anomaly score}
\rightarrow
\text{Decision}
}
$$

For J:

$$
\boxed{
1.2
\rightarrow
3.858
\rightarrow
0.806
\rightarrow
\text{Anomaly}
}
$$

For C:

$$
\boxed{
6.2
\rightarrow
3.858
\rightarrow
0.328
\rightarrow
\text{Normal}
}
$$

**One final important point:** the original \(s(x,n)\) score above is the theoretical Isolation Forest score. Scikit-learn exposes a different score convention through `score_samples()`/`decision_function()`, so when we reach Python implementation, we'll explicitly map the theory to sklearn rather than mixing the two conventions.


# Summary


Absolutely. Before implementation, here is the **complete flow of everything we learned in Isolation Forest**, including the small concepts in the correct order.

# Isolation Forest — Complete Theory Flow

### 1. Basic Intuition

Isolation Forest is based on one simple observation:

> **Anomalies are easier to isolate than normal points.**

If a point is far away from the main group, a random split can separate it quickly.

```text
Dense normal points          Isolated anomaly

A B C D E                    A B C D E        J
A B C D E                    A B C D E        ↑
                                                easy to isolate
```

Therefore:

$$
\boxed{\text{Shorter path} \rightarrow \text{more anomalous}}
$$

---

### 2. Why anomalies are easier to isolate

Normal points usually exist in **dense regions**.

So random splits keep leaving other normal points together.

An anomaly is usually in a **sparse region**, so fewer splits are needed to separate it.

```text
Normal point:
many neighboring points
        ↓
many splits
        ↓
long path

Anomaly:
few/no neighboring points
        ↓
few splits
        ↓
short path
```

---

### 3. Isolation Tree 🌳

An **Isolation Tree** is the individual tree inside the forest.

We start with all observations:

```text
A B C D E F G H I J
```

Then recursively split them until branches reach a stopping condition.

Each split consists of:

$$
\boxed{\text{Random Feature}+\text{Random Threshold}}
$$

Unlike a normal decision tree, we don't search for the "best" split.

The purpose is simply to **isolate observations**.

---

### 4. Random Feature Selection

At every node that needs splitting:

```text
Randomly select a feature
        ↓
Randomly select a threshold
        ↓
Split
```

For our 3-feature example:

```text
F1
F2
F3
```

A tree could do:

```text
Root       → F1
Child      → F3
Grandchild → F2
```

Or:

```text
Root       → F1
Child      → F1
Grandchild → F3
```

### Important:

**The same feature can be selected again.**

There is no rule that says:

> "Once F1 is used, don't use F1 again."

---

### 5. Random Split Selection

After selecting a feature, we randomly choose a **threshold**.

Suppose:

```text
F1 values:

2, 4, 6, 8, 20
```

Select F1.

Its range is:

$$
2 \rightarrow 20
$$

Suppose random threshold:

$$
10
$$

Then:

```text
F1 < 10
     /    \
 2 4 6 8   20
```

The threshold doesn't necessarily have to be an existing data value.

It can be something like:

$$
7.3
$$

---

### 6. Recursive Splitting

After a split, we don't stop the entire tree.

Each child is considered independently.

```text
             All points
              /       \
          group 1     group 2
           /  \         /  \
          ... ...      ... ...
```

If a branch still contains multiple separable points:

$$
\boxed{\text{keep splitting}}
$$

If only one point remains:

$$
\boxed{\text{stop}}
$$

Other stopping conditions can also exist, such as reaching the maximum tree depth.

---

### 7. Path Length

Now we ask:

> **How many splits did a point need before it was isolated?**

That is its **path length**.

For example:

```text
Root
 ↓
F1 < 20?
 ↓
J
```

J required one split:

$$
\boxed{h(J)=1}
$$

Another point might require:

```text
Root
 ↓
split 1
 ↓
split 2
 ↓
split 3
 ↓
C
```

So:

$$
\boxed{h(C)=3}
$$

Therefore:

$$
\boxed{\text{Short path} \rightarrow \text{anomaly}}
$$

$$
\boxed{\text{Long path} \rightarrow \text{normal}}
$$

---

# 8. Why One Tree Is Not Enough

The splits are **random**.

Therefore, one tree can sometimes isolate a normal point quickly just by chance.

So we don't rely on one tree.

We build:

```text
        Dataset
           │
     ┌─────┼─────┐
     ↓     ↓     ↓
   Tree1 Tree2 Tree3
     ↓     ↓     ↓
   Tree4 Tree5 ...
```

This is the **Isolation Forest**.

---

# 9. Path Length Across the Forest

For each point, we record its path length in every tree.

For J:

| Tree   | Path |
| ------ | ---: |
| Tree 1 |    1 |
| Tree 2 |    1 |
| Tree 3 |    1 |
| Tree 4 |    2 |
| Tree 5 |    1 |

Then calculate the average:

$$
E[h(J)]
=
\frac{1+1+1+2+1}{5}
$$

$$
\boxed{E[h(J)]=1.2}
$$

This tells us that J is consistently isolated very quickly.

---

# 10. Reference Path Length \(c(n)\)

We can't interpret an average path length by itself.

For example:

```text
Path = 3
```

might mean different things for different dataset sizes.

So Isolation Forest uses a normalization/reference value:

$$
\boxed{c(n)}
$$

For \(n>2\):

$$
c(n)
=
2H(n-1)
-
\frac{2(n-1)}{n}
$$

where:

$$
H(n-1)
=
\sum_{i=1}^{n-1}\frac{1}{i}
$$

For our 10-point dataset:

$$
c(10)\approx3.858
$$

This represents the **expected path length of an unsuccessful search in a random binary search tree** and acts as the normalization reference.

---

# 11. Anomaly Score

Now we compare the observed average path with the reference.

The theoretical Isolation Forest score is:

$$
s(x,n)
=
2^{-\frac{E[h(x)]}{c(n)}}
$$

For J:

$$
E[h(J)]=1.2
$$

and:

$$
c(10)=3.858
$$

Therefore:

$$
s(J,10)
=
2^{-\frac{1.2}{3.858}}
$$

$$
\boxed{s(J,10)\approx0.806}
$$

High score → more anomalous.

---

# 12. Compare With a Normal Point

For C, suppose:

$$
E[h(C)]=6.2
$$

Then:

$$
s(C,10)
=
2^{-\frac{6.2}{3.858}}
$$

$$
\boxed{s(C,10)\approx0.328}
$$

So:

| Point | Average Path | Anomaly Score |
| ----- | -----------: | ------------: |
| J     |          1.2 |     **0.806** |
| C     |          6.2 |     **0.328** |

Therefore:

```text
J → short path → high score → anomalous

C → long path → low score → normal
```

---

# 13. Complete Isolation Forest Flow

This is the **main flow you should remember**:

```text
                 DATASET
                    ↓
          Build many isolation trees
                    ↓
            ┌──── Tree 1 ────┐
            │                │
            │ Random Feature │
            │       ↓        │
            │ Random Split   │
            │       ↓        │
            │ Recursive split│
            │       ↓        │
            │ Path lengths   │
            └────────────────┘
                    ↓
              Tree 2, Tree 3...
                    ↓
          Path length for each point
                    ↓
          Average path length E[h(x)]
                    ↓
             Reference c(n)
                    ↓
             Anomaly Score
                    ↓
          ┌─────────┴─────────┐
          ↓                   ↓
      High score          Low score
          ↓                   ↓
       Anomaly              Normal
```

### The single most important chain

$$
\boxed{
\text{Random Feature}
\rightarrow
\text{Random Threshold}
\rightarrow
\text{Recursive Splitting}
\rightarrow
\text{Isolation}
\rightarrow
\text{Path Length}
\rightarrow
\text{Many Trees}
\rightarrow
\text{Average Path Length}
\rightarrow
c(n)
\rightarrow
\text{Anomaly Score}
\rightarrow
\text{Decision}
}
$$

And one important implementation distinction we'll need to handle carefully:

> **The theoretical score \(s(x,n)\) above and scikit-learn's `score_samples()` use different score conventions.** We'll map them explicitly when implementing rather than treating them as the same number.


# 10. Python Implementation 🐍

Now we'll implement the **same Isolation Forest idea in Python** using `scikit-learn`.

We'll first use a small dataset so we can connect the code directly to the theory we just learned.

### Step 1 — Import

```python
import numpy as np
from sklearn.ensemble import IsolationForest
```

### Step 2 — Create the dataset

```python
X = np.array([
    [10, 20, 100],   # A
    [11, 21, 101],   # B
    [12, 19, 99],    # C
    [13, 22, 102],   # D
    [14, 20, 100],   # E
    [15, 23, 103],   # F
    [11, 20, 98],    # G
    [13, 21, 101],   # H
    [14, 22, 102],   # I
    [30, 50, 150]    # J → unusual
])
```

Here:

* 10 rows = 10 observations
* 3 columns = 3 features
* We have **no labels** saying which point is anomalous.

That's exactly the type of situation Isolation Forest is designed for.

---

## Step 3 — Create the model

```python
model = IsolationForest(
    n_estimators=100,
    random_state=42
)
```

The important parameter here is:

```python
n_estimators=100
```

which means:

> Build **100 isolation trees**.

So conceptually:

```text
             Dataset
                ↓
     ┌──────────┼──────────┐
     ↓          ↓          ↓
  Tree 1     Tree 2     Tree 3
     ↓          ↓          ↓
    ...        ...        ...
     ↓
  Tree 100
                ↓
       Combine their results
```

---

## Step 4 — Train the forest

```python
model.fit(X)
```

Notice something important:

There is **no `y`**.

We aren't giving the model:

```python
model.fit(X, y)
```

because we don't have anomaly labels.

The model learns the isolation structure directly from `X`.

---

## Step 5 — Get anomaly predictions

```python
predictions = model.predict(X)
```

Scikit-learn returns:

```text
 1  → normal
-1  → anomaly
```

So:

```python
print(predictions)
```

might give something like:

```text
[ 1  1  1  1  1  1  1  1  1 -1]
```

Meaning:

```text
A → normal
B → normal
C → normal
D → normal
E → normal
F → normal
G → normal
H → normal
I → normal
J → anomaly
```

The exact result can depend on the model parameters and random seed.

---

## Step 6 — Get the actual anomaly scores

This is where we connect back to our mathematical discussion.

```python
scores = model.score_samples(X)
```

Important:

**Scikit-learn's `score_samples()` is not the same numerical convention as the theoretical \(s(x,n)\) we calculated earlier.**

For sklearn:

$$
\boxed{\text{More negative} \rightarrow \text{more abnormal}}
$$

So don't expect:

```text
theory:
0.806 → anomaly
0.328 → normal
```

to appear directly from:

```python
model.score_samples(X)
```

Instead, sklearn uses its own score convention.

---

## Step 7 — Put everything into a table

```python
for i, (prediction, score) in enumerate(zip(predictions, scores)):
    label = "Anomaly" if prediction == -1 else "Normal"
    print(i, label, score)
```

Conceptually, you'll get:

```text
Point    Prediction    Score
A        Normal        ...
B        Normal        ...
C        Normal        ...
D        Normal        ...
E        Normal        ...
F        Normal        ...
G        Normal        ...
H        Normal        ...
I        Normal        ...
J        Anomaly       ...
```

### Connect this directly to what we learned

The code is hiding the entire process we studied:

```text
model.fit(X)
      ↓
100 random isolation trees
      ↓
random feature selection
      ↓
random threshold selection
      ↓
recursive splitting
      ↓
path length for each point
      ↓
combine across trees
      ↓
score
      ↓
model.predict(X)
      ↓
Normal / Anomaly
```

So `fit()` isn't simply "memorizing the data."

It is constructing the **forest of random isolation trees** that we learned mathematically.


In [4]:
import numpy as np
from sklearn.ensemble import IsolationForest

X = np.array([
    [10, 20, 100],   # A
    [11, 21, 101],   # B
    [12, 19, 99],    # C
    [13, 22, 102],   # D
    [14, 20, 100],   # E
    [15, 23, 103],   # F
    [11, 20, 98],    # G
    [13, 21, 101],   # H
    [14, 22, 102],   # I
    [30, 50, 150]    # J → unusual
])

model = IsolationForest(
    n_estimators=100,
    random_state=42
)

model.fit(X)

predictions = model.predict(X)  # +1 Normal; -1 Anomaly
scores = model.score_samples(X) # Actual Anomaly score but here more negative more abnormal convention is used

for i , (prediction , score) in enumerate(zip(predictions , scores)):
    label = "Anomaly" if prediction == -1 else "Normal"
    print(i , label, score)

0 Normal -0.4354843278118496
1 Normal -0.4108862342449814
2 Normal -0.4824026742727405
3 Normal -0.40573511281579355
4 Normal -0.4220835094229189
5 Anomaly -0.5382324577010179
6 Normal -0.4522881859989721
7 Normal -0.38214941417233345
8 Normal -0.40728892634936326
9 Anomaly -0.7965860114830284


### 1. `n_estimators`

```python
IsolationForest(n_estimators=100)
```

* Controls **number of trees** in the forest.
* `100` → 100 independent isolation trees.
* More trees → generally more stable results, but more computation.
* It **does not control tree depth**.

$$
\boxed{\texttt{n\_estimators}=\text{number of trees}}
$$


### 2. `max_samples`

Controls **how many data points are used to build each tree**.

Example:

```python
IsolationForest(
    n_estimators=100,
    max_samples=256
)
```

If your dataset has 1000 rows:

$$
\boxed{\text{Each tree uses 256 samples}}
$$

rather than all 1000.

Why?

* Smaller `max_samples` → faster trees, more randomness.
* Larger `max_samples` → each tree sees more data, potentially more stable.
* If the dataset is smaller than `max_samples`, sklearn typically uses all available samples.

$$
\boxed{\texttt{max\_samples}=\text{number of samples used per tree}}
$$

**Key distinction:**

* `n_estimators` → **How many trees?**
* `max_samples` → **How many rows does each tree see?**


## 3. `contamination` — Example + Necessary Points

Suppose we have **100 data points**, and Isolation Forest gives each point an anomaly score.

Assume we set:

```python
model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)
```

### What does `0.05` mean?

$$
0.05 = 5\%
$$

So:

$$
100 \times 0.05 = 5
$$

The model will use a threshold such that approximately **5% of observations are classified as anomalies**.

Conceptually:

```text
100 data points
       ↓
Isolation Forest
       ↓
Anomaly scores
       ↓
Rank from less unusual → more unusual
       ↓
   ┌───────────────┐
   │ Top ~5 points │ → Anomaly
   └───────────────┘
       ↓
Remaining → Normal
```

For example:

| Point | Unusualness    |
| ----- | -------------- |
| P1    | Low            |
| P2    | Low            |
| P3    | Low            |
| ...   | ...            |
| P94   | Moderate       |
| P95   | High           |
| P96   | High           |
| P97   | Very high      |
| P98   | Very high      |
| P99   | Very high      |
| P100  | Extremely high |

With `contamination=0.05`, roughly the most unusual **5 points** would be classified as anomalies.

### Important: it does NOT change tree construction

`contamination` does **not** control:

* which rows are sampled
* which feature is selected
* where random splits occur
* number of trees

Those are controlled by other parts of Isolation Forest.

It mainly affects the **final cutoff**:

$$
\text{Anomaly Score} \rightarrow \boxed{\text{Threshold}} \rightarrow \text{Normal / Anomaly}
$$

### Necessary points to remember

1. **Range:** contamination is a proportion, e.g. `0.01`, `0.05`, `0.10`.
2. `0.05` = approximately **5%** expected contamination.
3. Higher contamination → **more points** classified as anomalies.
4. Lower contamination → **fewer points** classified as anomalies.
5. Choose it using **domain knowledge or a reasonable estimate** of the anomaly rate.
6. It affects the **decision threshold**, not how the isolation trees are constructed.
7. **Do not assume the chosen percentage represents the true number of anomalies**; it is an assumption used for classification.

**Memory shortcut:**

> `contamination` = **"How much anomaly do I expect in my dataset?"**


### 4. `max_features`

`max_features` controls **how many features (columns) each tree is allowed to use** when building the Isolation Forest.

Example: suppose your dataset has **5 features**:

```text
F1   F2   F3   F4   F5
```

If:

```python
max_features=1.0
```

→ each tree can use **all 5 features**.

If:

```python
max_features=0.6
```

→ each tree uses approximately **60% of the features**, so about 3 features.

### Why use it?

It adds more randomness to the forest.

```text
Dataset
   ↓
Tree 1 → F1, F3, F5
Tree 2 → F2, F3, F4
Tree 3 → F1, F2, F5
   ↓
Different trees see different feature subsets
   ↓
More diverse trees
```

### Don't confuse it with `max_samples`

| Parameter      | Randomly controls                           |
| -------------- | ------------------------------------------- |
| `max_samples`  | **Rows** used by each tree                  |
| `max_features` | **Columns/features** available to each tree |

**Memory shortcut:**

> `max_samples` → **How many data points?**
> `max_features` → **How many features?**


### 5. `max_depth`

Controls the **maximum depth of each isolation tree**.

```python
max_depth=5
```

→ Tree can make at most **5 levels of splits**.

* Higher depth → more splitting, more precise isolation.
* Lower depth → fewer splits.
* In sklearn, it is usually **left at the default**.

**Remember:** `max_depth` → **How deep can the tree grow?**


That completes the **Important Hyperparameters** section of `02_Isolation_Forest.ipynb`.

### Quick recap

| Parameter       | Controls                                             |
| --------------- | ---------------------------------------------------- |
| `n_estimators`  | Number of trees                                      |
| `max_samples`   | Number of rows used per tree                         |
| `contamination` | Expected proportion of anomalies / decision cutoff   |
| `max_features`  | Number/proportion of features available to each tree |
| `max_depth`     | Maximum tree depth                                   |
| `random_state`  | Randomness and reproducibility                       |

So the **Isolation Forest notebook is now complete**:

**Intuition → Trees → Randomness → Path Length → Score → Numerical Example → Python → Hyperparameters → Recap.**
